# CARLA Python API - Lab2

This notebook gives a step-by-step introduction to the CARLA Python API for controlling a simulation.

Lab goals:
- build a stable client-server connection;
- configure map, weather, and execution mode;
- add and manage actors in the scene.

Official documentation: [CARLA Python API](https://carla.readthedocs.io/en/latest/python_api/).

In [1]:
import carla
import time

## 1. Connect to the CARLA server

The first step is creating the `client`, which is the remote access point to the simulator.

`carla.Client(host, port)` does not start a new simulation: it connects to an already running CARLA server.

> Note: `Client` is an RPC proxy. Every API call sends a request to the server and receives a response.

In [2]:
client = carla.Client("localhost", 2000)

The connection is not immediate: network handshake, world loading, and latency can take a few seconds.

For this reason, set a suitable timeout before expensive operations such as `load_world`.

In [3]:
client.set_timeout(1)
client.load_world("Town01") # a time consuming operation

RuntimeError: time-out of 1000ms while waiting for the simulator, make sure the simulator is ready and connected to localhost:2000

In [6]:
client.set_timeout(15)
client.load_world("Town01")

In [7]:
print(f"Client version: {client.get_client_version()}")
print(f"Server version: {client.get_server_version()}")

Client version: 0.9.15
Server version: 0.9.15


## 2. Configure the simulation

After connecting, the `client` can manage the simulation context: map, weather, and other global settings.

### Change map

Loading a new map requires a full world reload. Waiting a few seconds is expected.

Additional maps can be downloaded from [GitHub Releases](https://github.com/carla-simulator/carla/releases).

Useful references:
- [Maps and navigation](https://carla.readthedocs.io/en/latest/core_map/)
- [`carla.Client.get_available_maps`](https://carla.readthedocs.io/en/latest/python_api/#carla.Client.get_available_maps)
- [`carla.Client.load_world`](https://carla.readthedocs.io/en/latest/python_api/#carla.Client.load_world)

In [8]:
client.get_available_maps()

['/Game/Carla/Maps/Town01',
 '/Game/Carla/Maps/Town01_Opt',
 '/Game/Carla/Maps/Town02',
 '/Game/Carla/Maps/Town02_Opt',
 '/Game/Carla/Maps/Town03',
 '/Game/Carla/Maps/Town03_Opt',
 '/Game/Carla/Maps/Town04',
 '/Game/Carla/Maps/Town04_Opt',
 '/Game/Carla/Maps/Town05',
 '/Game/Carla/Maps/Town05_Opt',
 '/Game/Carla/Maps/Town10HD',
 '/Game/Carla/Maps/Town10HD_Opt']

`Town10HD_Opt` e una versione alleggerita di `Town10HD`: mantiene buon dettaglio visivo con un costo computazionale piu basso.

In laboratorio e spesso una scelta pratica quando si vogliono test rapidi e stabili.

In [11]:
client.load_world("Town10HD_Opt")

### Set weather

`carla.WeatherParameters` includes presets that are useful for repeatable tests. Changing weather affects lighting, reflections, and visibility, and can therefore affect sensor perception as well.

Useful references:
- [Weather (core world)](https://carla.readthedocs.io/en/latest/core_world/#weather)
- [`carla.WeatherParameters`](https://carla.readthedocs.io/en/latest/python_api/#carla.WeatherParameters)
- [`carla.World.set_weather`](https://carla.readthedocs.io/en/latest/python_api/#carla.World.set_weather)

In [25]:
world = client.get_world()
world.set_weather(carla.WeatherParameters.WetNight)

In [26]:
weathers = [
    carla.WeatherParameters.CloudyNoon,
    carla.WeatherParameters.MidRainSunset,
    carla.WeatherParameters.WetNight,
    carla.WeatherParameters.HardRainNoon,
    carla.WeatherParameters.ClearSunset,
    carla.WeatherParameters.SoftRainNight,
    carla.WeatherParameters.Default,
]

for weather in weathers:
    world.set_weather(weather)
    print(f"Weather set to {weather}")
    time.sleep(5)

Weather set to WeatherParameters(cloudiness=60.000000, precipitation=0.000000, precipitation_deposits=0.000000, wind_intensity=10.000000, sun_azimuth_angle=-1.000000, sun_altitude_angle=45.000000, fog_density=3.000000, fog_distance=0.750000, fog_falloff=0.100000, wetness=0.000000, scattering_intensity=1.000000, mie_scattering_scale=0.030000, rayleigh_scattering_scale=0.033100, dust_storm=0.000000)
Weather set to WeatherParameters(cloudiness=60.000000, precipitation=60.000000, precipitation_deposits=60.000000, wind_intensity=60.000000, sun_azimuth_angle=-1.000000, sun_altitude_angle=15.000000, fog_density=3.000000, fog_distance=0.750000, fog_falloff=0.100000, wetness=0.000000, scattering_intensity=1.000000, mie_scattering_scale=0.030000, rayleigh_scattering_scale=0.033100, dust_storm=0.000000)


KeyboardInterrupt: 

### Synchronous vs asynchronous mode

In **synchronous** mode, the server advances only when the client calls `world.tick()`. This is useful for controlled experiments and reproducible results.

In **asynchronous** mode, the server advances on its own at the highest possible speed. This is useful for quick demos, but frame-by-frame control is lower.

Practical rule:
- use **synchronous** mode when you need to coordinate logic, sensors, or data collection;
- use **asynchronous** mode when you mainly want to observe the simulation in real time.

Useful references:
- [Synchrony and timestep](https://carla.readthedocs.io/en/latest/adv_synchrony_timestep/)
- [`carla.WorldSettings`](https://carla.readthedocs.io/en/latest/python_api/#carla.WorldSettings)
- [`carla.World.tick`](https://carla.readthedocs.io/en/latest/python_api/#carla.World.tick)

![Sync vs Async](./sync.png)

In [17]:
world = client.get_world()
settings = world.get_settings()
settings.synchronous_mode = True
world.apply_settings(settings)

17523

In [ ]:
count = 0
while True:
    world.tick()
    time.sleep(0.1)
    count += 1
    print(f"Ticked the server {count}", end="\r")

In [20]:
settings.synchronous_mode = False
world.apply_settings(settings)
client.reload_world()

## 3. Configuration exercises

These exercises are designed to consolidate the concepts above without introducing new topics.

### Guided exercise

Implement a script named `scenario_config.py` that:
1. connects to the server;
2. loads `Town10HD_Opt`;
3. applies at least 4 weather presets in sequence (with a 3-5 second pause);
4. switches to synchronous mode for 100 ticks;
5. switches back to asynchronous mode and reloads the world.

Required output: print each state transition to console (map, weather, mode, tick count).

### Additional exercises

1. Timeout handling: test `set_timeout(0.5)`, `5`, `15` and report which operations fail or succeed.
2. Sync/async comparison: measure how many ticks happen in 10 seconds in both modes.
3. Weather sequence: create a function that receives a list of presets and applies them in a continuous loop.

In [3]:
import carla
import time

weathers = [
    carla.WeatherParameters.CloudyNoon,
    carla.WeatherParameters.MidRainSunset,
    carla.WeatherParameters.WetNight,
    carla.WeatherParameters.HardRainNoon,
    carla.WeatherParameters.Default,
]

client = carla.Client("localhost", 2000)

client.set_timeout(15)
client.load_world("Town10HD_Opt")
print("Map loaded")
world = client.get_world()
world.set_weather(carla.WeatherParameters.CloudyNoon)
settings = world.get_settings()

for weather in weathers:
    world.set_weather(weather)
    print(f"Weather set to {weather}")
    time.sleep(5)

settings.synchronous_mode = True
print("Setting synchrounous mode")
world.apply_settings(settings)

countS = 0
while countS <= 100:
    world.tick()
    time.sleep(0.1)
    countS += 1
    print(f"Ticked the server {countS}", end="\r")

settings.synchronous_mode = False
world.apply_settings(settings)
print("Setting asynchrounous mode")
client.reload_world()


Map loaded
Weather set to WeatherParameters(cloudiness=60.000000, precipitation=0.000000, precipitation_deposits=0.000000, wind_intensity=10.000000, sun_azimuth_angle=-1.000000, sun_altitude_angle=45.000000, fog_density=3.000000, fog_distance=0.750000, fog_falloff=0.100000, wetness=0.000000, scattering_intensity=1.000000, mie_scattering_scale=0.030000, rayleigh_scattering_scale=0.033100, dust_storm=0.000000)
Weather set to WeatherParameters(cloudiness=60.000000, precipitation=60.000000, precipitation_deposits=60.000000, wind_intensity=60.000000, sun_azimuth_angle=-1.000000, sun_altitude_angle=15.000000, fog_density=3.000000, fog_distance=0.750000, fog_falloff=0.100000, wetness=0.000000, scattering_intensity=1.000000, mie_scattering_scale=0.030000, rayleigh_scattering_scale=0.033100, dust_storm=0.000000)
Weather set to WeatherParameters(cloudiness=5.000000, precipitation=0.000000, precipitation_deposits=50.000000, wind_intensity=10.000000, sun_azimuth_angle=-1.000000, sun_altitude_angle

## 4. Add actors to the simulation

### Coordinate system

Actors use `carla.Transform`, made of:
- `Location(x, y, z)` in meters;
- `Rotation(pitch, yaw, roll)` in degrees.

Understanding position and orientation is essential: small errors on `yaw` or `z` can cause invalid spawns or poor camera viewpoints.

In [38]:
spectator = world.get_spectator()
spectator.set_transform(
    carla.Transform(
        carla.Location(x=0, y=10, z=1),
        carla.Rotation(yaw=0, pitch=0, roll=0)
    )
)

### Spawn points

Spawn points are predefined map transforms where it is recommended to place actors (vehicles, pedestrians, sensors).

Cycling through them with the spectator is a quick way to understand scene geometry before spawning real actors.

Useful references:
- [`carla.Map.get_spawn_points`](https://carla.readthedocs.io/en/latest/python_api/#carla.Map.get_spawn_points)
- [`carla.World.get_map`](https://carla.readthedocs.io/en/latest/python_api/#carla.World.get_map)

In [21]:
world = client.get_world()
spawn_points = world.get_map().get_spawn_points()
spectator = world.get_spectator()
print(spawn_points.count)

for spawn_point in spawn_points:
    spectator.set_transform(spawn_point)

    print(spawn_point.location, end="\r")
    time.sleep(5)

<built-in method count of list object at 0x000001345CC801C8>


KeyboardInterrupt: 

### Blueprints and actors

The blueprint library contains the available actor models. Each blueprint has an `id` and `tags` that are useful for filtering what you need.

Useful references:
- [`carla.World.get_blueprint_library`](https://carla.readthedocs.io/en/latest/python_api/#carla.World.get_blueprint_library)
- [`carla.BlueprintLibrary`](https://carla.readthedocs.io/en/latest/python_api/#carla.BlueprintLibrary)
- [`carla.ActorBlueprint`](https://carla.readthedocs.io/en/latest/python_api/#carla.ActorBlueprint)

In [34]:
blue_prints = world.get_blueprint_library().filter('*')

for blue_print in blue_prints:
    print(f"Id: {blue_print.id}, Tags: {blue_print.tags}")

Id: vehicle.nissan.micra, Tags: ['vehicle', 'nissan', 'micra']
Id: vehicle.audi.a2, Tags: ['vehicle', 'audi', 'a2']
Id: static.prop.plantpot04, Tags: ['static', 'prop', 'plantpot04']
Id: static.prop.bench02, Tags: ['static', 'bench02', 'prop']
Id: vehicle.mercedes.coupe_2020, Tags: ['vehicle', 'mercedes', 'coupe_2020']
Id: vehicle.audi.tt, Tags: ['vehicle', 'audi', 'tt']
Id: vehicle.ford.ambulance, Tags: ['vehicle', 'ford', 'ambulance']
Id: vehicle.bmw.grandtourer, Tags: ['vehicle', 'bmw', 'grandtourer']
Id: vehicle.harley-davidson.low_rider, Tags: ['vehicle', 'harley-davidson', 'low_rider']
Id: walker.pedestrian.0037, Tags: ['walker', '0037', 'pedestrian']
Id: vehicle.micro.microlino, Tags: ['vehicle', 'micro', 'microlino']
Id: vehicle.carlamotors.firetruck, Tags: ['vehicle', 'carlamotors', 'firetruck']
Id: walker.pedestrian.0038, Tags: ['walker', 'pedestrian', '0038']
Id: static.prop.trashcan05, Tags: ['static', 'prop', 'trashcan05']
Id: vehicle.carlamotors.carlacola, Tags: ['vehicle

### Spawn a vehicle

`world.try_spawn_actor(...)` returns `None` if spawning fails (for example, if the spawn point is occupied).

Best practice: always check the result before calling methods such as `set_autopilot(True)` or `destroy()`.

Useful references:
- [`carla.World.try_spawn_actor`](https://carla.readthedocs.io/en/latest/python_api/#carla.World.try_spawn_actor)
- [`carla.Vehicle.set_autopilot`](https://carla.readthedocs.io/en/latest/python_api/#carla.Vehicle.set_autopilot)
- [`carla.Actor.destroy`](https://carla.readthedocs.io/en/latest/python_api/#carla.Actor.destroy)



In [48]:
vehicle = world.try_spawn_actor(blue_prints[1], spawn_points[1])
if vehicle is None:
    print("Spawn failed: spawn point occupied or invalid.")
else:
    print(f"Spawned vehicle: {vehicle.type_id}")

Spawned vehicle: vehicle.audi.a2


In [49]:
if vehicle is not None:
    vehicle.set_autopilot(True)
else:
    print("Autopilot not set: no vehicle available.")

In [13]:
if vehicle is not None:
    vehicle.destroy()
    vehicle = None
    print("Vehicle destroyed.")
else:
    print("Nothing to destroy.")

Nothing to destroy.


### Actor exercises

1. Robust spawn: try 50 random spawn points and count successful/failed spawns.
2. Blueprint filtering: print only blueprints with `vehicle` in tags and choose one specific model.
3. Cleanup: create a script that safely destroys all vehicles spawned by your script.

In [30]:
client.reload_world()

In [ ]:
blue_prints = world.get_blueprint_library().filter('vehicle.*')

for blue_print in blue_prints:
    print(f"Id: {blue_print.id}, Tags: {blue_print.tags}")

In [ ]:
client.reload_world()
import random
world = client.get_world()
blue_prints = world.get_blueprint_library().filter('vehicle.*')
spawn_points = world.get_map().get_spawn_points()
spectator = world.get_spectator()

count = 0
countError = 0
countSucc = 0
while count < 50:
    vehicle = world.try_spawn_actor(blue_prints[random.randint(0, 40)], spawn_points[random.randint(0, 100)])
    if vehicle is None:
        print("Spawn failed: spawn point occupied or invalid.")
        count += 1
        countSucc += 1
    else:
        print(f"Spawned vehicle: {vehicle.type_id}")
        count += 1
        countError += 1

print("numero di errori" + str(countError))

TypeError: unsupported operand type(s) for -: 'builtin_function_or_method' and 'int'

In [ ]:
import random
world = client.get_world()
blue_prints = world.get_blueprint_library().filter('vehicle.*')
spawn_points = world.get_map().get_spawn_points()
spectator = world.get_spectator()

count = 0
countError = 0
while count < 50:
    vehicle = world.try_spawn_actor(blue_prints[0], spawn_points[random.randint(0, 100)])
    if vehicle is None:
        print("Spawn failed: spawn point occupied or invalid.")
        count += 1
    else:
        print(f"Spawned vehicle: {vehicle.type_id}")
        count += 1
        countError += 1

print("numero di errori" + str(countError))